2.1

In [9]:
# 问题1：输出特征图尺寸
print("\n【问题1】输出特征图尺寸计算")
print("-" * 40)
print("已知条件：")
print("  输入尺寸: C_in = 3, H_in = 32, W_in = 32")
print("  卷积核: 16个, 每个大小 = 3 × 5 × 5")
print("  填充 P = 2, 步幅 S = 2")
print()
print("计算公式：")
print("  H_out = ⌊(H_in + 2P - K) / S⌋ + 1")
print("  W_out = ⌊(W_in + 2P - K) / S⌋ + 1")
print()
print("代入数值：")
print(f"  H_out = ⌊(32 + 2×2 - 5) / 2⌋ + 1")
print(f"       = ⌊(32 + 4 - 5) / 2⌋ + 1")
print(f"       = ⌊31 / 2⌋ + 1")
print(f"       = 15 + 1 = 16")
print()
print("结果：")
print("  输出特征图尺寸: 16 × 16 × 16 (通道数×高×宽)")

# 问题2：乘法次数
print("\n【问题2】单个输出通道的一个像素值的乘法次数")
print("-" * 40)
print("分析：")
print("  单个卷积核大小: 3 × 5 × 5 = 75 个权重")
print("  每个权重需要与输入对应位置做1次乘法")
print()
print("结果：")
print("  乘法次数 = 卷积核参数数量 = 75")


【问题1】输出特征图尺寸计算
----------------------------------------
已知条件：
  输入尺寸: C_in = 3, H_in = 32, W_in = 32
  卷积核: 16个, 每个大小 = 3 × 5 × 5
  填充 P = 2, 步幅 S = 2

计算公式：
  H_out = ⌊(H_in + 2P - K) / S⌋ + 1
  W_out = ⌊(W_in + 2P - K) / S⌋ + 1

代入数值：
  H_out = ⌊(32 + 2×2 - 5) / 2⌋ + 1
       = ⌊(32 + 4 - 5) / 2⌋ + 1
       = ⌊31 / 2⌋ + 1
       = 15 + 1 = 16

结果：
  输出特征图尺寸: 16 × 16 × 16 (通道数×高×宽)

【问题2】单个输出通道的一个像素值的乘法次数
----------------------------------------
分析：
  单个卷积核大小: 3 × 5 × 5 = 75 个权重
  每个权重需要与输入对应位置做1次乘法

结果：
  乘法次数 = 卷积核参数数量 = 75


2.2

In [3]:
import numpy as np

def max_pool2d_forward(x, kernel_size, stride=1, padding=0):
    # 转换为浮点类型以支持-inf填充
    x = np.asarray(x, dtype=np.float64)
    
    # 处理输入维度
    original_shape = x.shape
    if len(original_shape) == 2:
        x = x.reshape(1, 1, original_shape[0], original_shape[1])
        n, c, h, w = x.shape
    elif len(original_shape) == 3:
        x = x.reshape(1, original_shape[0], original_shape[1], original_shape[2])
        n, c, h, w = x.shape
    else:
        n, c, h, w = x.shape
    
    # 处理参数
    if isinstance(kernel_size, int):
        kh = kw = kernel_size
    else:
        kh, kw = kernel_size
    
    if isinstance(stride, int):
        sh = sw = stride
    else:
        sh, sw = stride
    
    if isinstance(padding, int):
        ph = pw = padding
    else:
        ph, pw = padding
    
    # 应用填充
    x_padded = np.pad(x, ((0, 0), (0, 0), (ph, ph), (pw, pw)), 
                      mode='constant', constant_values=-np.inf)
    
    # 计算输出尺寸
    out_h = (h + 2 * ph - kh) // sh + 1
    out_w = (w + 2 * pw - kw) // sw + 1
    
    # 初始化输出
    output = np.zeros((n, c, out_h, out_w))
    
    # 执行最大池化
    for i in range(out_h):
        for j in range(out_w):
            h_start = i * sh
            h_end = h_start + kh
            w_start = j * sw
            w_end = w_start + kw
            
            window = x_padded[:, :, h_start:h_end, w_start:w_end]
            output[:, :, i, j] = np.max(window, axis=(2, 3))
    
    # 恢复原始维度
    if len(original_shape) == 2:
        output = output.reshape(out_h, out_w)
    elif len(original_shape) == 3:
        output = output.reshape(c, out_h, out_w)
    
    return output


# 测试代码
if __name__ == "__main__":
    # 测试1: 2D输入
    print("=" * 50)
    print("测试1: 2D输入 (4x4矩阵)")
    x1 = np.array([
        [1, 2, 3, 4],
        [5, 6, 7, 8],
        [9, 10, 11, 12],
        [13, 14, 15, 16]
    ])
    out1 = max_pool2d_forward(x1, kernel_size=2, stride=2, padding=0)
    print("输入:\n", x1)
    print("输出 (kernel=2, stride=2):\n", out1)
    
    # 测试2: 带padding
    print("\n" + "=" * 50)
    print("测试2: 2D输入，带padding")
    x2 = np.array([
        [1, 2, 3],
        [4, 5, 6],
        [7, 8, 9]
    ])
    out2 = max_pool2d_forward(x2, kernel_size=3, stride=1, padding=1)
    print("输入:\n", x2)
    print("输出 (kernel=3, stride=1, padding=1):\n", out2)
    
    # 测试3: 3D输入 (C, H, W)
    print("\n" + "=" * 50)
    print("测试3: 3D输入 (2通道)")
    x3 = np.array([
        [[1, 2, 3, 4],
         [5, 6, 7, 8],
         [9, 10, 11, 12],
         [13, 14, 15, 16]],
        [[16, 15, 14, 13],
         [12, 11, 10, 9],
         [8, 7, 6, 5],
         [4, 3, 2, 1]]
    ])
    out3 = max_pool2d_forward(x3, kernel_size=2, stride=2, padding=0)
    print("输出形状:", out3.shape)
    print("输出通道0:\n", out3[0])
    print("输出通道1:\n", out3[1])

测试1: 2D输入 (4x4矩阵)
输入:
 [[ 1  2  3  4]
 [ 5  6  7  8]
 [ 9 10 11 12]
 [13 14 15 16]]
输出 (kernel=2, stride=2):
 [[ 6.  8.]
 [14. 16.]]

测试2: 2D输入，带padding
输入:
 [[1 2 3]
 [4 5 6]
 [7 8 9]]
输出 (kernel=3, stride=1, padding=1):
 [[5. 6. 6.]
 [8. 9. 9.]
 [8. 9. 9.]]

测试3: 3D输入 (2通道)
输出形状: (2, 2, 2)
输出通道0:
 [[ 6.  8.]
 [14. 16.]]
输出通道1:
 [[16. 14.]
 [ 8.  6.]]


3.1

In [15]:
# 问题1：5x5卷积参数量
print("\n【问题1】单个5×5卷积层参数量")
print("-" * 40)
print("已知条件：")
print("  输入通道数 = C")
print("  输出通道数 = C")
print("  卷积核大小: 5 × 5")
print()
print("计算公式：")
print("  参数量 = 输入通道数 × 输出通道数 × 核高 × 核宽")
print()
print("代入数值：")
print("  参数量 = C × C × 5 × 5")
print("         = C × C × 25")
print("         = 25C²")
print()
print("结果：")
print("  参数量 = 25C²")

# 问题2：两个3x3卷积参数量
print("\n【问题2】两个串联的3×3卷积层总参数量")
print("-" * 40)
print("已知条件：")
print("  每层输入通道数 = C")
print("  每层输出通道数 = C")
print("  卷积核大小: 3 × 3")
print()
print("计算公式：")
print("  单层参数量 = 输入通道数 × 输出通道数 × 核高 × 核宽")
print("            = C × C × 3 × 3 = 9C²")
print("  总参数量 = 第一层 + 第二层 = 9C² + 9C²")
print()
print("结果：")
print("  总参数量 = 18C²")



【问题1】单个5×5卷积层参数量
----------------------------------------
已知条件：
  输入通道数 = C
  输出通道数 = C
  卷积核大小: 5 × 5

计算公式：
  参数量 = 输入通道数 × 输出通道数 × 核高 × 核宽

代入数值：
  参数量 = C × C × 5 × 5
         = C × C × 25
         = 25C²

结果：
  参数量 = 25C²

【问题2】两个串联的3×3卷积层总参数量
----------------------------------------
已知条件：
  每层输入通道数 = C
  每层输出通道数 = C
  卷积核大小: 3 × 3

计算公式：
  单层参数量 = 输入通道数 × 输出通道数 × 核高 × 核宽
            = C × C × 3 × 3 = 9C²
  总参数量 = 第一层 + 第二层 = 9C² + 9C²

结果：
  总参数量 = 18C²


3.2

In [4]:
import torch
import torch.nn as nn

class NiNBlock(nn.Module):
    
    def __init__(self, in_channels, out_channels, kernel_size, stride, padding):
        super(NiNBlock, self).__init__()
        self.block = nn.Sequential(
            # 普通卷积层
            nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding),
            nn.ReLU(),
            # 1x1卷积层
            nn.Conv2d(out_channels, out_channels, kernel_size=1),
            nn.ReLU(),
            # 另一个1x1卷积层
            nn.Conv2d(out_channels, out_channels, kernel_size=1),
            nn.ReLU()
        )
    
    def forward(self, x):
        return self.block(x)


# 测试代码
if __name__ == "__main__":
    print("=" * 50)
    print("NiN块测试")
    
    # 创建NiN块实例
    nin_block = NiNBlock(
        in_channels=3,
        out_channels=96,
        kernel_size=3,
        stride=1,
        padding=1
    )
    
    # 创建随机输入 (batch=1, channels=3, height=224, width=224)
    x = torch.randn(1, 3, 224, 224)
    
    # 前向传播
    y = nin_block(x)
    
    print(f"输入形状: {x.shape}")
    print(f"输出形状: {y.shape}")
    print(f"\nNiN块结构:\n{nin_block.block}")

NiN块测试
输入形状: torch.Size([1, 3, 224, 224])
输出形状: torch.Size([1, 96, 224, 224])

NiN块结构:
Sequential(
  (0): Conv2d(3, 96, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (1): ReLU()
  (2): Conv2d(96, 96, kernel_size=(1, 1), stride=(1, 1))
  (3): ReLU()
  (4): Conv2d(96, 96, kernel_size=(1, 1), stride=(1, 1))
  (5): ReLU()
)


4.1

In [12]:

import math
from fractions import Fraction

x = [2, 4, 6, 8]
gamma = 2
beta = 1
epsilon = 0

print("已知条件：")
print(f"  x = {x}")
print(f"  γ = {gamma}")
print(f"  β = {beta}")
print(f"  ε = {epsilon}")
print()

# 步骤1：计算均值
print("【步骤1】计算均值 μ")
print("-" * 40)
mu = sum(x) / len(x)
print(f"  μ = (2 + 4 + 6 + 8) / 4")
print(f"    = 20 / 4 = {mu}")
print()

# 步骤2：计算方差
print("【步骤2】计算方差 σ²")
print("-" * 40)
variance = sum((xi - mu) ** 2 for xi in x) / len(x)
print(f"  σ² = [(2-5)² + (4-5)² + (6-5)² + (8-5)²] / 4")
print(f"     = [(-3)² + (-1)² + (1)² + (3)²] / 4")
print(f"     = [9 + 1 + 1 + 9] / 4")
print(f"     = 20 / 4 = {variance}")
print()

# 步骤3：标准化
print("【步骤3】标准化计算")
print("-" * 40)
print("公式: x̂ᵢ = (xᵢ - μ) / √(σ² + ε)")
print(f"  √(σ² + ε) = √{variance} = √5")
print()
print("计算：")
x_hat_values = []
for i, xi in enumerate(x, 1):
    x_hat = (xi - mu) / math.sqrt(variance + epsilon)
    x_hat_values.append(x_hat)
    print(f"  x̂_{i} = ({xi} - 5) / √5 = {Fraction(xi-5, 1)}/√5")

print()

# 步骤4：缩放和平移
print("【步骤4】缩放和平移")
print("-" * 40)
print("公式: yᵢ = γ·x̂ᵢ + β")
print()
print("计算结果：")
y_values = []
for i, x_hat in enumerate(x_hat_values, 1):
    y = gamma * x_hat + beta
    y_values.append(y)
    # 生成分数形式
    num = (x[i-1] - 5) * gamma
    print(f"  y_{i} = 2 × ({Fraction(x[i-1]-5, 1)}/√5) + 1 = {num}/√5 + 1")

print()
print("【最终结果】")
print("-" * 40)
print("精确形式：")
print(f"  y₁ = -6/√5 + 1")
print(f"  y₂ = -2/√5 + 1")
print(f"  y₃ = 2/√5 + 1")
print(f"  y₄ = 6/√5 + 1")
print()
print("数值近似：")
for i, y in enumerate(y_values, 1):
    print(f"  y_{i} = {y:.4f}")

已知条件：
  x = [2, 4, 6, 8]
  γ = 2
  β = 1
  ε = 0

【步骤1】计算均值 μ
----------------------------------------
  μ = (2 + 4 + 6 + 8) / 4
    = 20 / 4 = 5.0

【步骤2】计算方差 σ²
----------------------------------------
  σ² = [(2-5)² + (4-5)² + (6-5)² + (8-5)²] / 4
     = [(-3)² + (-1)² + (1)² + (3)²] / 4
     = [9 + 1 + 1 + 9] / 4
     = 20 / 4 = 5.0

【步骤3】标准化计算
----------------------------------------
公式: x̂ᵢ = (xᵢ - μ) / √(σ² + ε)
  √(σ² + ε) = √5.0 = √5

计算：
  x̂_1 = (2 - 5) / √5 = -3/√5
  x̂_2 = (4 - 5) / √5 = -1/√5
  x̂_3 = (6 - 5) / √5 = 1/√5
  x̂_4 = (8 - 5) / √5 = 3/√5

【步骤4】缩放和平移
----------------------------------------
公式: yᵢ = γ·x̂ᵢ + β

计算结果：
  y_1 = 2 × (-3/√5) + 1 = -6/√5 + 1
  y_2 = 2 × (-1/√5) + 1 = -2/√5 + 1
  y_3 = 2 × (1/√5) + 1 = 2/√5 + 1
  y_4 = 2 × (3/√5) + 1 = 6/√5 + 1

【最终结果】
----------------------------------------
精确形式：
  y₁ = -6/√5 + 1
  y₂ = -2/√5 + 1
  y₃ = 2/√5 + 1
  y₄ = 6/√5 + 1

数值近似：
  y_1 = -1.6833
  y_2 = 0.1056
  y_3 = 1.8944
  y_4 = 3.6833


4.2

In [5]:
import torch
import torch.nn as nn

class Residual(nn.Module):
    def __init__(self, in_channels, out_channels, use_1x1conv=False, stride=1):
        super(Residual, self).__init__()
        
        # 第一个卷积层
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1)
        self.bn1 = nn.BatchNorm2d(out_channels)
        
        # 第二个卷积层
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        # 1x1卷积调整残差连接
        if use_1x1conv:
            self.conv3 = nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride)
        else:
            self.conv3 = None
        
        self.relu = nn.ReLU()
    
    def forward(self, x):
        # 主路径
        y = self.relu(self.bn1(self.conv1(x)))
        y = self.bn2(self.conv2(y))
        
        # 残差连接
        if self.conv3 is not None:
            x = self.conv3(x)
        
        # 相加后激活
        y = self.relu(y + x)
        return y


# 测试代码
if __name__ == "__main__":
    print("=" * 50)
    print("残差块测试")
    
    # 测试1: 输入输出通道相同，无1x1卷积
    print("\n测试1: 输入输出通道相同，无1x1卷积")
    res_block1 = Residual(in_channels=64, out_channels=64, use_1x1conv=False)
    x1 = torch.randn(1, 64, 32, 32)
    y1 = res_block1(x1)
    print(f"输入形状: {x1.shape}")
    print(f"输出形状: {y1.shape}")
    
    # 测试2: 输入输出通道不同，使用1x1卷积
    print("\n测试2: 输入输出通道不同，使用1x1卷积")
    res_block2 = Residual(in_channels=32, out_channels=64, use_1x1conv=True, stride=2)
    x2 = torch.randn(1, 32, 32, 32)
    y2 = res_block2(x2)
    print(f"输入形状: {x2.shape}")
    print(f"输出形状: {y2.shape}")
    
    # 打印第一个残差块的结构
    print(f"\n残差块结构:\n{res_block1}")

残差块测试

测试1: 输入输出通道相同，无1x1卷积
输入形状: torch.Size([1, 64, 32, 32])
输出形状: torch.Size([1, 64, 32, 32])

测试2: 输入输出通道不同，使用1x1卷积
输入形状: torch.Size([1, 32, 32, 32])
输出形状: torch.Size([1, 64, 16, 16])

残差块结构:
Residual(
  (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU()
)


5.1

1. 为什么底层特征提取层用小学习率，顶层输出层用大学习率？

底层特征提取层：这些层学习的是通用的、可迁移的特征（如边缘、纹理、颜色等）。在源数据集上已经训练得很好，直接用于目标任务通常也很有效。用较小的学习率可以：保留预训练学到的通用特征，防止对源数据的知识造成破坏性更新，避免在小目标数据集上过拟合

顶层输出层：这些层学习的是与具体任务相关的语义特征。新目标数据集的类别分布与源数据集不同，因此需要重新训练。用较大的学习率可以：快速适应新任务的特定类别，学习目标数据集特有的特征表示

2. 目标数据集很小且与源数据集非常相似时的微调策略

冻结大部分底层特征提取层：将除最后几层外的所有层参数固定

只微调最后的全连接层：重新初始化最后的输出层并用较大学习率训练

使用较小的学习率：如果决定微调部分特征提取层，使用比正常训练小10-100倍的学习率

增加数据增强：即使数据集相似，也要使用更强的数据增强防止过拟合

使用Dropout和权重衰减：增加正则化强度

早停（Early Stopping）：监控验证集性能，防止过拟合

5.2

In [6]:
import torchvision.transforms as transforms
from PIL import Image
import torch

def create_augmentation_pipeline():
    pipeline = transforms.Compose([
        # 随机裁剪，面积比例0.08~1.0，缩放到224x224
        transforms.RandomResizedCrop(
            size=224,
            scale=(0.08, 1.0),
            ratio=(1.0, 1.0)  # 保持正方形裁剪
        ),
        # 50%概率水平翻转
        transforms.RandomHorizontalFlip(p=0.5),
        # 随机改变亮度、对比度、饱和度
        transforms.ColorJitter(
            brightness=0.5,
            contrast=0.5,
            saturation=0.5
        ),
        # 转换为张量
        transforms.ToTensor()
    ])
    
    return pipeline


# 测试代码
if __name__ == "__main__":
    print("=" * 50)
    print("图像增广管道测试")
    
    # 创建增广管道
    aug_pipeline = create_augmentation_pipeline()
    
    print(f"增广管道:\n{aug_pipeline}")
    
    # 创建测试图像 (使用随机RGB图像模拟)
    # 实际使用中: img = Image.open('your_image.jpg')
    test_img = Image.new('RGB', (500, 500), color=(100, 150, 200))
    
    # 应用增广
    augmented_tensor = aug_pipeline(test_img)
    
    print(f"\n输入图像大小: 500x500")
    print(f"输出张量形状: {augmented_tensor.shape}")
    print(f"输出张量值范围: [{augmented_tensor.min():.4f}, {augmented_tensor.max():.4f}]")
    
    # 演示多次应用不同的增广结果
    print("\n" + "=" * 50)
    print("多次应用增广结果对比:")
    
    for i in range(3):
        aug_img = aug_pipeline(test_img)
        print(f"第{i+1}次增广 - 形状: {aug_img.shape}, 均值: {aug_img.mean():.4f}")

图像增广管道测试
增广管道:
Compose(
    RandomResizedCrop(size=(224, 224), scale=(0.08, 1.0), ratio=(1.0, 1.0), interpolation=bilinear, antialias=True)
    RandomHorizontalFlip(p=0.5)
    ColorJitter(brightness=(0.5, 1.5), contrast=(0.5, 1.5), saturation=(0.5, 1.5), hue=None)
    ToTensor()
)

输入图像大小: 500x500
输出张量形状: torch.Size([3, 224, 224])
输出张量值范围: [0.4471, 0.8000]

多次应用增广结果对比:
第1次增广 - 形状: torch.Size([3, 224, 224]), 均值: 0.7464
第2次增广 - 形状: torch.Size([3, 224, 224]), 均值: 0.4157
第3次增广 - 形状: torch.Size([3, 224, 224]), 均值: 0.6157


6.1

In [13]:

from fractions import Fraction
# 定义边界框
A = [10, 10, 50, 50]  # [x1, y1, x2, y2]
B = [30, 30, 70, 70]

print("已知条件：")
print(f"  真实框 A = {A}")
print(f"  预测框 B = {B}")
print("  格式: [左上角x, 左上角y, 右下角x, 右下角y]")
print()

# 步骤1：计算交集
print("【步骤1】计算交集区域")
print("-" * 40)
x_left = max(A[0], B[0])
y_top = max(A[1], B[1])
x_right = min(A[2], B[2])
y_bottom = min(A[3], B[3])

print(f"  交集左上角: (max({A[0]}, {B[0]}), max({A[1]}, {B[1]})) = ({x_left}, {y_top})")
print(f"  交集右下角: (min({A[2]}, {B[2]}), min({A[3]}, {B[3]})) = ({x_right}, {y_bottom})")
print()

width = x_right - x_left
height = y_bottom - y_top
print(f"  交集宽度 = {x_right} - {x_left} = {width}")
print(f"  交集高度 = {y_bottom} - {y_top} = {height}")

intersection = width * height
print(f"  交集面积 = {width} × {height} = {intersection}")
print()

# 步骤2：计算各自面积
print("【步骤2】计算各自面积")
print("-" * 40)
area_A = (A[2] - A[0]) * (A[3] - A[1])
area_B = (B[2] - B[0]) * (B[3] - B[1])

print(f"  Area_A = ({A[2]} - {A[0]}) × ({A[3]} - {A[1]}) = {A[2]-A[0]} × {A[3]-A[1]} = {area_A}")
print(f"  Area_B = ({B[2]} - {B[0]}) × ({B[3]} - {B[1]}) = {B[2]-B[0]} × {B[3]-B[1]} = {area_B}")
print()

# 步骤3：计算并集
print("【步骤3】计算并集面积")
print("-" * 40)
union = area_A + area_B - intersection
print(f"  Union = Area_A + Area_B - Intersection")
print(f"        = {area_A} + {area_B} - {intersection}")
print(f"        = {union}")
print()

# 步骤4：计算IoU
print("【步骤4】计算IoU")
print("-" * 40)
iou = intersection / union
iou_fraction = Fraction(intersection, union)
print(f"  IoU = Intersection / Union")
print(f"      = {intersection} / {union}")
print(f"      = {iou_fraction}")
print(f"      ≈ {iou:.6f}")
print()

print("【最终结果】")
print("-" * 40)
print(f"  IoU = {iou_fraction} ≈ {iou:.6f}")

已知条件：
  真实框 A = [10, 10, 50, 50]
  预测框 B = [30, 30, 70, 70]
  格式: [左上角x, 左上角y, 右下角x, 右下角y]

【步骤1】计算交集区域
----------------------------------------
  交集左上角: (max(10, 30), max(10, 30)) = (30, 30)
  交集右下角: (min(50, 70), min(50, 70)) = (50, 50)

  交集宽度 = 50 - 30 = 20
  交集高度 = 50 - 30 = 20
  交集面积 = 20 × 20 = 400

【步骤2】计算各自面积
----------------------------------------
  Area_A = (50 - 10) × (50 - 10) = 40 × 40 = 1600
  Area_B = (70 - 30) × (70 - 30) = 40 × 40 = 1600

【步骤3】计算并集面积
----------------------------------------
  Union = Area_A + Area_B - Intersection
        = 1600 + 1600 - 400
        = 2800

【步骤4】计算IoU
----------------------------------------
  IoU = Intersection / Union
      = 400 / 2800
      = 1/7
      ≈ 0.142857

【最终结果】
----------------------------------------
  IoU = 1/7 ≈ 0.142857


6.2

In [7]:
import torch
import torch.nn.functional as F

class LabelSmoothingCrossEntropy:
    def __init__(self, epsilon=0.1, reduction='mean'):
        self.epsilon = epsilon
        self.reduction = reduction
    
    def __call__(self, predictions, targets):
        n, c = predictions.shape
        
        # 平滑后的目标概率分布
        # 真实类别概率: 1 - epsilon
        # 其他类别概率: epsilon / (C - 1)
        smooth_targets = torch.zeros_like(predictions)
        smooth_targets.fill_(self.epsilon / (c - 1))
        smooth_targets.scatter_(1, targets.unsqueeze(1), 1 - self.epsilon)
        
        # 计算log softmax
        log_probs = F.log_softmax(predictions, dim=1)
        
        # 计算损失: -sum(y_true * log(y_pred))
        loss = -torch.sum(smooth_targets * log_probs, dim=1)
        
        # 应用reduction
        if self.reduction == 'mean':
            return loss.mean()
        elif self.reduction == 'sum':
            return loss.sum()
        else:
            return loss


def test_label_smoothing():
    print("=" * 50)
    print("标签平滑交叉熵损失测试")
    
    # 创建模拟数据
    torch.manual_seed(42)
    n, c = 4, 5  # 4个样本，5个类别
    predictions = torch.randn(n, c)  # 随机logits
    targets = torch.tensor([0, 2, 1, 3])  # 真实标签
    
    print(f"预测logits形状: {predictions.shape}")
    print(f"真实标签: {targets}")
    print(f"\n预测logits:\n{predictions}")
    
    # 使用标准交叉熵作为对比
    standard_ce = F.cross_entropy(predictions, targets, reduction='mean')
    print(f"\n标准交叉熵损失: {standard_ce.item():.6f}")
    
    # 使用标签平滑
    epsilon = 0.1
    ls_ce = LabelSmoothingCrossEntropy(epsilon=epsilon, reduction='mean')
    smooth_loss = ls_ce(predictions, targets)
    print(f"标签平滑损失 (epsilon={epsilon}): {smooth_loss.item():.6f}")
    
    # 显示平滑后的目标分布
    smooth_targets = torch.zeros(n, c)
    smooth_targets.fill_(epsilon / (c - 1))
    for i, t in enumerate(targets):
        smooth_targets[i, t] = 1 - epsilon
    
    print(f"\n平滑后目标概率分布 (样本1): {smooth_targets[0].tolist()}")
    print(f"  真实类别概率: {1 - epsilon}")
    print(f"  其他类别概率: {epsilon / (c - 1)}")
    
    # 对比不同epsilon的影响
    print("\n" + "=" * 50)
    print("不同平滑因子影响对比:")
    for eps in [0.0, 0.05, 0.1, 0.2, 0.5]:
        ls_loss = LabelSmoothingCrossEntropy(epsilon=eps, reduction='mean')
        loss_val = ls_loss(predictions, targets)
        print(f"  epsilon={eps}: {loss_val.item():.6f}")


if __name__ == "__main__":
    test_label_smoothing()

标签平滑交叉熵损失测试
预测logits形状: torch.Size([4, 5])
真实标签: tensor([0, 2, 1, 3])

预测logits:
tensor([[ 1.9269,  1.4873,  0.9007, -2.1055, -0.7581],
        [ 1.0783,  0.8008,  1.6806,  0.3559, -0.6866],
        [-0.4934,  0.2415, -0.2316,  0.0418, -0.2516],
        [ 0.8599, -0.3097, -0.3957,  0.8034, -0.6216]])

标准交叉熵损失: 0.978850
标签平滑损失 (epsilon=0.1): 1.097218

平滑后目标概率分布 (样本1): [0.8999999761581421, 0.02500000037252903, 0.02500000037252903, 0.02500000037252903, 0.02500000037252903]
  真实类别概率: 0.9
  其他类别概率: 0.025

不同平滑因子影响对比:
  epsilon=0.0: 0.978850
  epsilon=0.05: 1.038034
  epsilon=0.1: 1.097218
  epsilon=0.2: 1.215587
  epsilon=0.5: 1.570693
